# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [29]:
import pandas as pd
import numpy as np

# Step 1: Data Loading and Inspection
df = pd.read_csv('AviationData.csv', encoding='latin-1', low_memory=False)

# Inspect NaNs and datatypes
print("Info")
print(df.info())

print("\nMissing Values")
print(df.isnull().sum())

print("\n Summary Statistics")
print(df.describe(include='all'))

# Show first few rows to understand structure
print("\n Head ")
print(df.head())

Info
<class 'pandas.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  str    
 1   Investigation.Type      88889 non-null  str    
 2   Accident.Number         88889 non-null  str    
 3   Event.Date              88889 non-null  str    
 4   Location                88837 non-null  str    
 5   Country                 88663 non-null  str    
 6   Latitude                34382 non-null  str    
 7   Longitude               34373 non-null  str    
 8   Airport.Code            50132 non-null  str    
 9   Airport.Name            52704 non-null  str    
 10  Injury.Severity         87889 non-null  str    
 11  Aircraft.damage         85695 non-null  str    
 12  Aircraft.Category       32287 non-null  str    
 13  Registration.Number     87507 non-null  str    
 14  Make                    88826 non-null  str 

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [40]:
# --- STEP 2: Data Cleaning & Filtering ---
# Standardize dates
df['Event.Date'] = pd.to_datetime(df['Event.Date'])
max_date = df['Event.Date'].max()
cutoff_date = max_date - pd.DateOffset(years=40)

# Imputation for Aircraft.Category before filtering to save data
# Standardize Make first to make matching easier
df['Make'] = df['Make'].str.upper().str.strip()
airplanes_makes = ['CESSNA', 'PIPER', 'BEECH', 'MOONEY', 'GRUMMAN', 'BELLANCA', 'BOEING', 'AIRBUS']
df.loc[df['Aircraft.Category'].isna() & df['Make'].isin(airplanes_makes), 'Aircraft.Category'] = 'Airplane'

# Apply filters
df_clean = df[
    (df['Aircraft.Category'] == 'Airplane') &
    (df['Amateur.Built'] == 'No') &
    (df['Event.Date'] >= cutoff_date)
].copy()

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [44]:
# --- STEP 3: Cleaning and Constructing Key Measurables ---
# Clean injury columns
injury_cols = ['Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured']
for col in injury_cols:
    df_clean[col] = df_clean[col].fillna(0)

# Calculate Metric: Total Passengers and Injury Rate
df_clean['Total.Passengers'] = df_clean[injury_cols].sum(axis=1)
# Use a safety check for division by zero
df_clean['Injury.Metric'] = np.where(df_clean['Total.Passengers'] > 0,
                                     (df_clean['Total.Fatal.Injuries'] + df_clean['Total.Serious.Injuries']) / df_clean['Total.Passengers'],
                                     0)



**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [43]:
# Standardize Aircraft.damage and create 'Is_Destroyed'
df_clean['Aircraft.damage'] = df_clean['Aircraft.damage'].str.strip().str.capitalize()
df_clean['Is_Destroyed'] = df_clean['Aircraft.damage'] == 'Destroyed'

In [36]:
print(df.columns)

Index(['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date',
       'Location', 'Country', 'Latitude', 'Longitude', 'Airport.Code',
       'Airport.Name', 'Injury.Severity', 'Aircraft.damage',
       'Aircraft.Category', 'Registration.Number', 'Make', 'Model',
       'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'FAR.Description',
       'Schedule', 'Purpose.of.flight', 'Air.carrier', 'Total.Fatal.Injuries',
       'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured',
       'Weather.Condition', 'Broad.phase.of.flight', 'Report.Status',
       'Publication.Date', 'Total.Occupants', 'Injury.Rate', 'is_destroyed'],
      dtype='str')


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [42]:
# --- STEP 4: Investigate Make Column ---
# Keep Makes with >= 50 occurrences
make_counts = df_clean['Make'].value_counts()
valid_makes = make_counts[make_counts >= 50].index
df_clean = df_clean[df_clean['Make'].isin(valid_makes)].copy()

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [45]:
# --- STEP 5: Inspect Model Column ---
# Drop rows where Model is NaN
df_clean = df_clean.dropna(subset=['Model'])
df_clean['Model'] = df_clean['Model'].str.upper().str.strip()
# Create unique identifier for plane type
df_clean['Plane.Type'] = df_clean['Make'] + " " + df_clean['Model']

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [46]:
cols_to_standardize = ['Engine.Type', 'Weather.Condition', 'Purpose.of.flight', 'Broad.phase.of.flight']
for col in cols_to_standardize:
    df_clean[col] = df_clean[col].str.strip().str.capitalize()

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [48]:
# Keep all columns with more than 20,000 non-nulls
non_null_counts = df_clean.count()
cols_to_keep = non_null_counts[non_null_counts > 20000].index
df_final = df_clean[cols_to_keep].copy()
# Double check the remaining columns
print(f"Remaining columns: {df_final.columns.tolist()}")


Remaining columns: ['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date', 'Location', 'Country', 'Latitude', 'Longitude', 'Airport.Code', 'Airport.Name', 'Injury.Severity', 'Aircraft.damage', 'Aircraft.Category', 'Registration.Number', 'Make', 'Model', 'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'Purpose.of.flight', 'Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured', 'Weather.Condition', 'Broad.phase.of.flight', 'Report.Status', 'Publication.Date', 'Total.Occupants', 'Injury.Rate', 'is_destroyed', 'Flight.Category', 'Total.Passengers', 'Injury.Metric', 'Is_Destroyed', 'Plane.Type']


In [49]:
# Final check for our key client metrics
print(df_final[['Is_Destroyed', 'Injury.Metric']].describe())

       Injury.Metric
count   53276.000000
mean        0.266925
std         0.426660
min         0.000000
25%         0.000000
50%         0.000000
75%         0.500000
max         1.000000


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [50]:
# Save the intermediate cleaned state
output_filename = 'Aviation_Safety_Cleaned_1983_2023.csv'

try:
    df_final.to_csv(output_filename, index=False)
    print(f"Successfully saved cleaned data to: {output_filename}")
except Exception as e:
    print(f"Error saving file: {e}")

Successfully saved cleaned data to: Aviation_Safety_Cleaned_1983_2023.csv
